# PreTest 02: Regression แบบครบวงจร — การทำนายค่าเบี้ยประกันสุขภาพ
### จากข้อมูลดิบ → ทำความสะอาด → Visualization → เตรียมข้อมูล → เทรน 3 โมเดล → วัดผล → เลือกโมเดล

> **ที่มา:** ดัดแปลงและขยายจาก Kaggle Notebook
> [Linear Regression Tutorial](https://www.kaggle.com/code/sudhirnl7/linear-regression-tutorial) โดย sudhirnl7
> สิ่งที่เพิ่ม/แก้: คำอธิบายภาษาไทยทุกขั้นตอน, ตรวจ/ลบข้อมูลซ้ำ, Metrics ครบ (MAE, RMSE, R²) ทั้งสเกล log และสเกลเงินจริง,
> เพิ่มโมเดล **Decision Tree** และ **Random Forest**, Cross-Validation, ตารางเปรียบเทียบ 3 โมเดล, แก้โค้ดให้ใช้กับ seaborn/sklearn ปัจจุบัน
>
> **ชุดข้อมูล (Dataset):** [Medical Cost Personal Datasets](https://www.kaggle.com/datasets/mirichoi0218/insurance) — ค่าประกันสุขภาพรายปีของคนในสหรัฐฯ 1,338 คน

---

## ภาพรวมขั้นตอน

| ขั้น | งาน | คำถามที่ต้องตอบ |
|---|---|---|
| 1 | กำหนดโจทย์ | ทำนายอะไร? วัดผลด้วยอะไร? |
| 2–3 | โหลดและสำรวจข้อมูล | แต่ละคอลัมน์คืออะไร? |
| 4 | ทำความสะอาด | ค่าหาย? ข้อมูลซ้ำ? Outlier? |
| 5 | EDA & Visualization | Target กระจายตัวอย่างไร? อะไรทำให้ค่าประกันแพง? |
| 6 | เตรียมข้อมูล | Encode → แปลง Target → Split |
| 7–9 | เทรนโมเดล 3 ตัว | Linear Regression, Decision Tree, Random Forest |
| 10 | เปรียบเทียบและเลือกโมเดล | ตัวไหนดีที่สุด เพราะอะไร? |

---
# ขั้นที่ 1: กำหนดโจทย์

- **เป้าหมาย:** ทำนาย **ค่าประกันสุขภาพต่อปี** (`charges`, หน่วย USD) จากข้อมูลส่วนตัวของผู้ทำประกัน
- **ประเภทปัญหา:** Target เป็นตัวเลขต่อเนื่อง → **Regression**
- **วัดผลด้วย:**

| Metric | สูตร (โดยย่อ) | ความหมาย | ดีเมื่อ |
|---|---|---|---|
| **MAE** | เฉลี่ยของ \|จริง − ทำนาย\| | ทำนายพลาดเฉลี่ยกี่ดอลลาร์ | ต่ำ |
| **RMSE** | √(เฉลี่ยของ (จริง − ทำนาย)²) | เหมือน MAE แต่ลงโทษการพลาดมากๆ หนักกว่า | ต่ำ |
| **R²** | 1 − (SSE / SST) | อธิบายความแปรปรวนของ Target ได้กี่ % | ใกล้ 1 |

---
# ขั้นที่ 2: นำเข้าไลบรารีและโหลดข้อมูล

---
# ขั้นที่ 3: สำรวจข้อมูลเบื้องต้น

| คอลัมน์ | ความหมาย | ชนิด |
|---|---|---|
| `age` | อายุผู้ทำประกัน | Numerical |
| `sex` | เพศ (male/female) | Categorical |
| `bmi` | ดัชนีมวลกาย (kg/m²) — ค่าตั้งแต่ 30 ขึ้นไปจัดเป็นภาวะอ้วน | Numerical |
| `children` | จำนวนบุตรที่อยู่ในประกัน (0–5) | Numerical (ค่าไม่ต่อเนื่อง) |
| `smoker` | สูบบุหรี่หรือไม่ (yes/no) | Categorical |
| `region` | ภูมิภาคที่อยู่ในสหรัฐฯ (4 ภาค) | Categorical |
| **`charges`** | **ค่าประกันต่อปี (USD) ← Target** | Numerical |

> **ข้อสังเกต:** `charges` มีตั้งแต่ ~1,100 ถึง ~63,800 ดอลลาร์ และค่าเฉลี่ย (13,270) **สูงกว่า** median (9,382) มาก → การกระจาย **เบ้ขวา**
> ผู้สูบบุหรี่มี 274 คน (~20%)

---
# ขั้นที่ 4: ทำความสะอาดข้อมูล

### 4.1 ค่าที่หาย (Missing Values)

> **ผลลัพธ์:** ไม่มีค่าหาย (ถ้ามี ให้ใช้เทคนิคจากบท 02-2 เช่นเติมด้วย median/mode)

### 4.2 ข้อมูลซ้ำ (Duplicates)

> **หมายเหตุ:** พบ 1 แถวที่ซ้ำกันทุกคอลัมน์ ซึ่งเป็นไปได้ยากในข้อมูลจริง (อายุ เพศ BMI ค่าประกันเท่ากันทุกค่า) จึงลบออก

### 4.3 ตรวจ Outlier

> **ตัดสินใจ: ไม่ลบ Outlier** เพราะ
> - `charges` สูง **เป็นค่าจริง** ของกลุ่มผู้สูบบุหรี่ (จะเห็นในขั้นที่ 5) ไม่ใช่ข้อมูลผิด — ถ้าลบ โมเดลจะทำนายคนกลุ่มนี้ไม่ได้
> - `bmi` สูงสุด ~53 ยังเป็นค่าที่เป็นไปได้ทางการแพทย์
>
> **หลักคิด:** ลบ Outlier เฉพาะเมื่อมั่นใจว่าเป็น **ข้อมูลผิดพลาด** เท่านั้น

---
# ขั้นที่ 5: EDA & Visualization

### 5.1 การกระจายของ Target (`charges`)

> **ข้อสังเกต:** ซ้าย: เบ้ขวามาก (skew ≈ 1.5) / ขวา: หลังใส่ log ใกล้ระฆังคว่ำมากขึ้น (skew ≈ −0.1)
> → เราจะ **ทำนาย `log(charges)`** เพราะ Linear Regression ทำงานได้ดีเมื่อ Target ไม่เบ้มาก แล้วค่อยแปลงกลับเป็นดอลลาร์ด้วย `exp()`

### 5.2 ตัวแปร Categorical กับค่าประกัน

> **ผู้สูบบุหรี่จ่ายแพงกว่าประมาณ 4 เท่า** (เฉลี่ย ~32,000 vs ~8,400) → `smoker` น่าจะเป็นตัวแปรสำคัญที่สุด
> เพศและภูมิภาคต่างกันไม่มาก

### 5.3 จำนวนบุตรกับค่าประกัน

### 5.4 อายุ / BMI กับค่าประกัน แยกตามการสูบบุหรี่ (กราฟสำคัญที่สุด)

> **Insight สำคัญ:**
> - **อายุ:** ค่าประกันเพิ่มตามอายุเป็นเส้นตรง แยกเป็น 3 "ชั้น" ตามสถานะสูบบุหรี่
> - **BMI:** สำหรับคน **ไม่สูบ** BMI แทบไม่มีผล แต่ **ผู้สูบที่ BMI ≥ 30 ค่าประกันกระโดดขึ้นไปอีกระดับ**
> - นี่คือ **Interaction** (ผลของ BMI ขึ้นกับการสูบบุหรี่) ซึ่ง Linear Regression ธรรมดา **จับไม่ได้** แต่ Tree จับได้ → จะเห็นผลในขั้นที่ 10

### 5.5 Correlation (ตัวแปรตัวเลข)

> **ข้อสังเกต:** `smoker_yes` สัมพันธ์กับ `charges` สูงมาก (~0.79) รองลงมาคือ `age` (~0.30) และ `bmi` (~0.20)
> ตัวแปรต้นด้วยกันเองไม่สัมพันธ์กันมาก → ไม่มีปัญหา Multicollinearity

---
# ขั้นที่ 6: เตรียมข้อมูลสำหรับโมเดล

### 6.1 One-Hot Encoding และ Dummy Variable Trap
- แปลง `sex`, `smoker`, `region`, `children` เป็นคอลัมน์ 0/1
- `drop_first=True` → ตัดหมวดแรกทิ้ง 1 คอลัมน์ เพราะรู้ค่าได้จากคอลัมน์ที่เหลือ
  (เช่น ถ้า `sex_male = 0` ก็รู้ว่าเป็น female) ถ้าไม่ตัด จะเกิด **Dummy Variable Trap** ทำให้ Linear Regression คำนวณไม่เสถียร
- `children` เป็นตัวเลขก็จริง แต่ต้นฉบับมองเป็นหมวดหมู่ (ผลของจำนวนบุตรอาจไม่เป็นเส้นตรง)

### 6.2 แปลง Target ด้วย log
(ต้นฉบับทดลอง Box-Cox ด้วย แต่ผลไม่ดีกว่า log จึงใช้ log ซึ่งง่ายกว่า)

### 6.3 แบ่ง Train / Test (70 / 30)

> **หมายเหตุ:** ข้อมูลชุดนี้ไม่ต้อง Scale เพราะ Linear Regression แบบไม่มี Regularization ไม่ได้รับผลจากสเกล และโมเดลกลุ่ม Tree ไม่ได้รับผลจากสเกล
> (ถ้าใช้ KNN, SVR, Ridge/Lasso ควร Scale)

### 6.4 ฟังก์ชันวัดผล (ใช้ซ้ำกับทุกโมเดล)
วัดทั้งในสเกล log (ที่โมเดลเรียน) และ **สเกลดอลลาร์จริง** (ที่คนเข้าใจ) ด้วยการแปลงกลับด้วย `np.exp`

---
# ขั้นที่ 7: โมเดลที่ 1 — Linear Regression

### หลักการ
หาเส้น (ระนาบ) ที่ใกล้ทุกจุดที่สุด: `ŷ = θ₀ + θ₁x₁ + θ₂x₂ + … + θₙxₙ`

- **Cost function (MSE):** `J(θ) = (1/m) Σ (ŷᵢ − yᵢ)²` → หาค่า θ ที่ทำให้ J ต่ำสุด
- **Normal Equation:** หาคำตอบได้โดยตรงด้วยสูตร `θ = (XᵀX)⁻¹ Xᵀy`

### 7.1 ภาพตัวอย่าง: เส้นตรง 1 ตัวแปร (BMI → charges)

### 7.2 คำนวณด้วยมือ (Normal Equation) เทียบกับ scikit-learn

> **ผลลัพธ์:** สองวิธีได้ค่าเท่ากัน → `LinearRegression().fit()` ก็คือการแก้สมการนี้ให้เรานั่นเอง

### 7.3 ตีความ Coefficient
Target เป็น log → coefficient θ หมายถึง **ค่าประกันเปลี่ยนไปประมาณ (e^θ − 1) × 100%**

> **ข้อสังเกต:** เช่น `smoker_yes` ≈ 1.55 → ผู้สูบจ่ายแพงกว่า ≈ +373% (ประมาณ 4.7 เท่า) / `age` ≈ 0.034 → อายุเพิ่ม 1 ปี ค่าประกันเพิ่ม ≈ 3.4%

### 7.4 วัดผล

> **ข้อสังเกต:** R² ในสเกล log ≈ 0.79 อยู่ในระดับดี แต่ **R² ในสเกลดอลลาร์ต่ำกว่ามาก** — เพราะโมเดลทำนายกลุ่ม "สูบบุหรี่และมีภาวะอ้วน" ที่ค่าประกันสูงมากได้ไม่ดี
> และความผิดพลาดในกลุ่มค่าสูงจะขยายใหญ่เมื่อแปลงกลับจาก log
> (การใส่ log ไม่ได้ดีขึ้นเสมอไป — ทดลองพิสูจน์เองในแบบฝึกหัดข้อ 1 ท้ายบท)

### 7.5 ตรวจสมมติฐานของ Linear Regression (Model Validation)
| สมมติฐาน | ตรวจด้วย | ถ้าผ่าน |
|---|---|---|
| ความสัมพันธ์เป็นเส้นตรง | Actual vs Predicted | จุดเรียงตามเส้นทแยง |
| Residual กระจายแบบปกติ, mean ≈ 0 | Histogram ของ Residual | รูประฆัง ศูนย์กลางที่ 0 |
| Normality | Q-Q plot | จุดอยู่บนเส้นตรง |
| Homoscedasticity (ความแปรปรวนคงที่) | Residual vs Predicted | กระจายเป็นแถบเท่ากัน ไม่เป็นรูปกรวย |

> **ผลการตรวจ:** กราฟ 1 มีจุดกลุ่มหนึ่งเบี่ยงจากเส้น, Residual เบ้ขวา, Q-Q ปลายบนโค้งออก
> → สมมติฐานไม่ผ่านทั้งหมด แปลว่าความสัมพันธ์จริง **ไม่เป็นเส้นตรงล้วน** (มี Interaction ระหว่าง smoker × bmi ตามที่เห็นในข้อ 5.4)
> → ควรทดลองโมเดลที่จับความสัมพันธ์ไม่เป็นเส้นตรงได้ ได้แก่ Tree

### 7.6 การวิเคราะห์ผล Linear Regression

| ประเด็น | ผลลัพธ์ | การตีความ |
|---|---|---|
| คำนวณเองตรงกับ sklearn หรือไม่ | θ จาก Normal Equation = θ จาก sklearn ทุกตัว | `fit()` คือการแก้สมการ θ = (XᵀX)⁻¹Xᵀy |
| Overfit หรือไม่ | R² (log) Train 0.761 vs Test 0.787 | ไม่ Overfit (Test สูงกว่าเล็กน้อยเป็นเรื่องปกติจากการสุ่มแบ่ง) |
| ทำนายพลาดเฉลี่ยเท่าใด | MAE **$4,309** | ≈ 32% ของค่าประกันเฉลี่ย ($13,270) — ค่อนข้างมาก |
| มีการทำนายพลาดหนักหรือไม่ | RMSE **$8,213** (เกือบ 2 เท่าของ MAE) | มีบางคนที่ทำนายพลาดหนักมาก (กลุ่มค่าประกันสูง) |
| อธิบายความแปรปรวนได้เพียงใด? | R² (log) 0.787 แต่ **R² ($) 0.533** | วัดเป็นดอลลาร์จริง อธิบายได้เพียงครึ่งเดียว |
| ปัจจัยไหนมีผล? | สูบบุหรี่ +373%, อายุ +3.4%/ปี, BMI +1.2%/หน่วย | จุดเด่น: บอกขนาดผลของแต่ละปัจจัยเป็นตัวเลขได้ |
| สมมติฐานผ่านหรือไม่ | Residual เบ้ขวา, Q-Q โค้งปลายบน | ความสัมพันธ์ไม่เป็นเส้นตรงล้วน |

> **สรุป:** ตีความง่ายที่สุด (บอกได้ว่าแต่ละปัจจัยเพิ่มค่าประกันกี่ %) แต่ **แม่นน้อย** เพราะจับ Interaction
> "สูบบุหรี่ × BMI ≥ 30" ไม่ได้ ระวังด้วยว่า **R² ในสเกล log สูงกว่าความแม่นยำที่แท้จริง** ต้องวัดในสเกลดอลลาร์เสมอ

---
# ขั้นที่ 8: โมเดลที่ 2 — Decision Tree Regressor

### หลักการ
แบ่งข้อมูลด้วยคำถาม Yes/No ไปเรื่อยๆ (เช่น *สูบบุหรี่?* → *BMI > 30?*) แต่ละใบ (leaf) ทำนายด้วย **ค่าเฉลี่ย** ของข้อมูลในใบนั้น
เลือกจุดแบ่งที่ทำให้ **MSE ภายในกลุ่มลูกต่ำที่สุด** → จับ Interaction ได้เองโดยธรรมชาติ

### 8.1 เลือก `max_depth` ด้วย Cross-Validation

> **ข้อสังเกต:** CV R² สูงสุดที่ `max_depth = 4` หลังจากนั้น Train ยังเพิ่ม แต่ CV ลดลง (Overfitting) — ไม่จำกัดความลึก Train R² ≈ 1.0 แต่ CV ต่ำที่สุด

> **ข้อสังเกต:** ต้นไม้เลือกแบ่งด้วย `smoker_yes` ก่อน แล้วใช้ `bmi` / `age` ต่อ — ตรงกับ Insight จาก EDA ทุกประการ

### 8.2 ผลบน Test Set

### 8.3 การวิเคราะห์ผล Decision Tree

| ประเด็น | ผลลัพธ์ | การตีความ |
|---|---|---|
| ไม่จำกัดความลึกเป็นอย่างไร? | Train R² **0.997** vs CV R² **0.644** | Overfit หนัก — ต่ำกว่า Linear Regression |
| ความลึกที่เลือก | `max_depth = 4` (CV R² สูงสุด 0.796) | เลือกจาก CV ไม่ใช่จาก Test |
| Overfit หรือไม่ (หลังจูน)? | R² (log) Train 0.816 vs Test 0.828 | ไม่ Overfit |
| ทำนายพลาดเฉลี่ยเท่าใด | MAE **$2,449** | ลดลง **43%** จาก Linear Regression ($4,309) |
| มีการทำนายพลาดหนักหรือไม่ | RMSE **$4,808** | ลดลงเกือบครึ่ง — ทำนายกลุ่มค่าประกันสูงได้ดีขึ้นมาก |
| อธิบายความแปรปรวนได้เพียงใด? | R² ($) **0.840** | เพิ่มจาก 0.533 |
| ต้นไม้ตัดสินอย่างไร? | แบ่งด้วย `smoker_yes` ก่อน แล้วจึง `bmi` / `age` | เรียนรู้ Interaction ที่ Linear Regression ทำไม่ได้ |

> **สรุป:** ต้นไม้เพียง 4 ชั้นแม่นกว่า Linear Regression มาก เพราะจับ "การสูบบุหรี่ × ภาวะอ้วน" ได้เอง และยังอ่านเป็นกฎได้
> แต่ต้อง **จำกัดความลึก** เสมอ มิฉะนั้นจะเกิด Overfitting อย่างรุนแรง

---
# ขั้นที่ 9: โมเดลที่ 3 — Random Forest Regressor

### หลักการ
สร้าง Decision Tree หลายร้อยต้นจากข้อมูลที่สุ่มต่างกัน แล้ว **เฉลี่ยคำทำนาย** → แม่นและเสถียรกว่าต้นเดียว

### 9.1 จูน Hyperparameter ด้วย GridSearchCV

> **ข้อสังเกต:** `smoker_yes` สำคัญที่สุด รองลงมาอย่างใกล้เคียงคือ `age` และ `bmi` ตามลำดับ — เพศ ภูมิภาค จำนวนบุตร แทบไม่มีผล

### 9.2 ผลบน Test Set

### 9.3 การวิเคราะห์ผล Random Forest

| ประเด็น | ผลลัพธ์ | การตีความ |
|---|---|---|
| Hyperparameter ที่ดีที่สุด | `max_depth=5`, `min_samples_leaf=1` (CV R² 0.806) | ต้นไม้ไม่ต้องลึกมาก |
| Overfit หรือไม่ | R² (log) Train 0.843 vs Test 0.838 | ไม่ Overfit |
| ทำนายพลาดเฉลี่ยเท่าใด | MAE **$2,254** | ต่ำสุด — ดีกว่า Decision Tree ~8% (≈ 17% ของค่าเฉลี่ย) |
| มีการทำนายพลาดหนักหรือไม่ | RMSE **$4,657** | ต่ำสุด |
| อธิบายความแปรปรวนได้เพียงใด? | R² ($) **0.850** | สูงสุด |
| ปัจจัยสำคัญ | สูบบุหรี่ 0.53 > อายุ 0.40 > BMI 0.04 | เพศ ภูมิภาค จำนวนบุตร แทบไม่มีผล |

> **สรุป:** แม่นที่สุดในสามโมเดล แต่ดีกว่า Decision Tree **เพียงเล็กน้อย** (R² 0.850 vs 0.840)
> แลกกับการที่อ่านเป็นกฎไม่ได้ (เป็นค่าเฉลี่ยของต้นไม้ 300 ต้น)

---
# ขั้นที่ 10: เปรียบเทียบ 3 โมเดลและเลือกโมเดลที่ดีที่สุด

### 10.1 ตารางเปรียบเทียบ (Test Set)

### 10.2 กราฟ Actual vs Predicted (สเกลดอลลาร์)

> **ข้อสังเกต:** Linear Regression ทำนายกลุ่มผู้สูบที่ค่าประกันสูง (มุมขวาบน) **ต่ำกว่าจริง** มาก ส่วนโมเดล Tree ทั้งสองจุดเกาะเส้นทแยงดีกว่าชัดเจน

### 10.3 ใช้โมเดลทำนายลูกค้าใหม่
ข้อมูลใหม่ต้องผ่าน **ขั้นตอนเตรียมข้อมูลเดียวกัน** (One-Hot แล้วจัดคอลัมน์ให้ตรงกับตอนเทรน)

> **ข้อสังเกต:** ผู้ชายอายุ 40 ไม่สูบ: BMI 25 หรือ 34 ค่าประกันแทบไม่ต่างกัน (~$5,900–7,100)
> แต่ถ้า **สูบ + BMI 34** Linear Regression ทำนาย $31,091 ขณะที่ Tree ทำนาย $36,076–39,921 → Linear Regression ทำนายกลุ่มนี้ **ต่ำเกินจริง**

### 10.4 สรุปผลการเปรียบเทียบ

| โมเดล | MAE ($) | RMSE ($) | R² ($) | Overfit? | จุดเด่น |
|---|---|---|---|---|---|
| Linear Regression | 4,309 | 8,213 | 0.533 | ไม่ | ตีความง่ายที่สุด บอกผลเป็น % ได้ |
| Decision Tree (depth 4) | 2,449 | 4,808 | 0.840 | ไม่ | แม่นใกล้ RF และอ่านเป็นกฎได้ |
| Random Forest | **2,254** | **4,657** | **0.850** | ไม่ | แม่นที่สุด |

**ข้อค้นพบหลัก**
1. **โมเดล Tree ดีกว่า Linear Regression มาก** (R² 0.84–0.85 vs 0.53) เพราะข้อมูลมี Interaction "สูบบุหรี่ × BMI ≥ 30" ที่เส้นตรงจับไม่ได้
2. **Random Forest ดีกว่า Decision Tree เพียงเล็กน้อย** (MAE ต่างกันเพียง ~$200)
3. **RMSE ≈ 2 เท่าของ MAE ทุกโมเดล** → ยังมีบางคนที่ทำนายพลาดหนัก (ค่าประกันสูงมาก)
4. **ปัจจัยสำคัญ:** การสูบบุหรี่ > อายุ > BMI (BMI มีผลเฉพาะเมื่อสูบด้วย) ส่วนเพศ ภูมิภาค จำนวนบุตร แทบไม่มีผล
5. **R² ในสเกล log อาจทำให้เข้าใจผิดได้** — Linear Regression ได้ 0.787 (log) แต่ในสเกลดอลลาร์จริงเพียง 0.533

**คำแนะนำ**
- ต้องการความแม่นสูงสุด → **Random Forest**
- ต้องการอธิบายให้ลูกค้า/ผู้บริหารเข้าใจ → **Decision Tree (depth 4)** โดยเสียความแม่นยำเพียงเล็กน้อย
- หมายเหตุ: ดูแบบฝึกหัดข้อ 1: Linear Regression ที่ **ไม่ใส่ log + เพิ่ม Feature `smoker_obese`** ได้ R² ($) **0.858** ชนะทุกโมเดล!

> **ตัวอย่างการเขียนสรุปผลในข้อสอบ:**
> *"เลือก Random Forest เพราะได้ MAE ต่ำสุด ($2,254) และ R² สูงสุด (0.850) บน Test Set โดยไม่ Overfit (R² Train 0.843 vs Test 0.838)
> ดีกว่า Linear Regression (R² 0.533) อย่างชัดเจน เพราะจับความสัมพันธ์ระหว่างการสูบบุหรี่กับ BMI ได้
> ข้อจำกัดคือ RMSE ยังสูงเกือบ 2 เท่าของ MAE แสดงว่ายังทำนายพลาดหนักในบางราย"*

---
## รายการตรวจสอบสำหรับการสอบ (Checklist): Regression
- [ ] ตรวจค่าหาย, ข้อมูลซ้ำ, Outlier (และอธิบายว่าทำไมเก็บ/ลบ)
- [ ] ดูการกระจายของ Target — ถ้าเบ้มาก ทดลอง log
- [ ] EDA หาความสัมพันธ์ของแต่ละ Feature กับ Target
- [ ] One-Hot (`drop_first=True` สำหรับ Linear Regression)
- [ ] Split → เทรน 2–3 โมเดล → เลือก Hyperparameter ด้วย CV
- [ ] รายงาน MAE, RMSE, R² (ถ้าแปลง Target ต้องแปลงกลับก่อนรายงาน)
- [ ] Linear Regression: ตรวจ Residual plots
- [ ] สรุปเลือกโมเดลพร้อมเหตุผล

## แบบฝึกหัด
1. ทดลอง Linear Regression 4 แบบ แล้วเทียบ R² (ดอลลาร์):
   (ก) Target แบบ log (แบบในบทนี้) (ข) Target เป็นหน่วยดอลลาร์โดยไม่แปลง log
   (ค)–(ง) แบบ ก และ ข แต่เพิ่ม Feature ใหม่ `smoker_obese = smoker_yes × (bmi ≥ 30)` จาก Insight ในข้อ 5.4
2. เพิ่มโมเดลที่ 4: `KNeighborsRegressor` (ต้อง Scale ข้อมูลก่อน) แล้วเพิ่มลงในตารางเปรียบเทียบ

> **บทเรียนสำคัญจากแบบฝึกหัดนี้**
> 1. **log ไม่ได้ดีเสมอไป** — ต้นฉบับเลือก log เพราะ Target เบ้ แต่เมื่อวัดในสเกลดอลลาร์ การไม่ใส่ log กลับได้ผลดีกว่า → **ต้องทดลองและวัดผลในสเกลจริงเสมอ**
> 2. **Feature Engineering จาก EDA ทรงพลังมาก** — เพิ่มเพียง 1 คอลัมน์ Linear Regression ก็แม่นใกล้เคียงหรือดีกว่า Random Forest และยังตีความง่ายกว่า